# Census MTIS 目录传输单次修复

原固定版本的目录GET返回HTTP500，三个官方PDF已NAS冷恢复。本卡只重取同一个目录一次，若能精确绑定链接，才从已保存原字节解析三个年份制造业合计库存销售比及发布时间；0 SOXX结果、0拟合、0账户。

In [1]:
import datetime,hashlib,io,json,os,pathlib,re,shlex,subprocess,sys,tarfile,tempfile,urllib.parse
from bs4 import BeautifulSoup
from pypdf import PdfReader
import qlib
from qlib.data import D
from qlib.workflow import R
root=pathlib.Path.cwd() if (pathlib.Path.cwd()/'src').exists() else pathlib.Path.cwd().parents[1];sys.path.insert(0,str(root))
from src.data.nas_market_data import nas_api_call,archive_nas_recorder,_settings,_ssh
card=root/'research/stage1_census_mtis_index_transport_repair_20261023';prior=root/'research/stage1_census_mtis_manufacturer_ratio_source_pilot_20261023';snap=root/'data/snapshots/stage1_census_mtis_index_transport_repair_20261023';snap.mkdir(parents=True,exist_ok=True);p=json.loads((card/'plan.json').read_text());s=p['sources'];sha=lambda x:hashlib.sha256(pathlib.Path(x).read_bytes()).hexdigest();assert p['frozen'] and not (card/'run_state.json').exists()
assert sha(prior/'summary.json')==s['prior_summary_sha256'] and sha(prior/'result.json')==s['prior_result_sha256'] and sha(root/'data/snapshots/stage1_census_mtis_manufacturer_ratio_source_pilot_20261023/response_receipts.json')==s['prior_receipts_sha256'];old=json.loads((prior/'summary.json').read_text());assert old['source_NAS_cold_restore_verified'] and old['source_NAS_sha256']=='79fce1f4120782f84415ffeef3e42b59066887cd9825eca9da4bf91d9f2ae119'
old_snap=root/'data/snapshots/stage1_census_mtis_manufacturer_ratio_source_pilot_20261023'
for month,h in s['saved_PDF_SHA256'].items():assert sha(old_snap/(month+'.response'))==h
state={'status':'index_GET_started','plan_sha256':sha(card/'plan.json'),'GETs_started':1,'saved_PDF_reads':0,'model_fits':0,'native_accounts':0};(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
body=snap/'official_index.response';headers=snap/'official_index.headers';start=datetime.datetime.now(datetime.timezone.utc).isoformat();cmd=['curl','--http1.1','--location','--silent','--show-error','--connect-timeout','5','--max-time','20','--user-agent','Mozilla/5.0 (compatible; SOXL-Qlib-Research/1.0)','--dump-header',str(headers),'--output',str(body),'--write-out','%{http_code}|%{url_effective}',s['index_url']];call=subprocess.run(cmd,capture_output=True,text=True);parts=call.stdout.strip().split('|',1)
receipt={'url':s['index_url'],'started_utc':start,'ended_utc':datetime.datetime.now(datetime.timezone.utc).isoformat(),'curl_exit':call.returncode,'http_status':parts[0] if parts else None,'effective_url':parts[1] if len(parts)>1 else None,'error':call.stderr[:250] or None,'body_bytes':body.stat().st_size if body.exists() else 0,'body_sha256':sha(body) if body.exists() else None,'headers_bytes':headers.stat().st_size if headers.exists() else 0,'headers_sha256':sha(headers) if headers.exists() else None};(snap/'receipt.json').write_text(json.dumps(receipt,indent=2)+'\n')
links=set()
if receipt['http_status']=='200' and body.exists() and body.stat().st_size<=700000:
 soup=BeautifulSoup(body.read_bytes(),'html.parser');links={urllib.parse.urljoin(s['index_url'],a.get('href')) for a in soup.find_all('a') if a.get('href')}
months={'2011-07':'mtis1107.pdf','2018-07':'mtis1807.pdf','2025-07':'mtis2507.pdf'};case_results=[]
if all('https://www2.census.gov/mtis/historical/'+v in links for v in months.values()):
 qlib.init(provider_uri=str(root/'data/qlib/soxl'),region='us',kernels=1);calendar=[str(x)[:10] for x in D.calendar(start_time='2011-09-01',end_time='2025-10-31',freq='day')];mon=['JANUARY','FEBRUARY','MARCH','APRIL','MAY','JUNE','JULY','AUGUST','SEPTEMBER','OCTOBER','NOVEMBER','DECEMBER']
 for month,fname in months.items():
  fp=old_snap/(month+'.response');data=fp.read_bytes();state['saved_PDF_reads']+=1;(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n');c={'economic_month':month,'PDF_SHA256':sha(fp),'index_link_exact':True,'PDF_magic':data.startswith(b'%PDF'),'release_date':None,'release_time_ET':None,'first_Qlib_trade_after_release':None,'manufacturers_adjusted_current_ratio':None,'manufacturers_adjusted_previous_ratio':None,'status':'source_layout_unknown'}
  if c['PDF_magic']:
   t='\n'.join(pg.extract_text() or '' for pg in PdfReader(io.BytesIO(data)).pages);head=t[:1600].upper();date=re.search(r'(JANUARY|FEBRUARY|MARCH|APRIL|MAY|JUNE|JULY|AUGUST|SEPTEMBER|OCTOBER|NOVEMBER|DECEMBER)\s+(\d{1,2}),\s+(20\d{2})',head);time_ok=bool(re.search(r'10:00\s*(?:A\.?M\.?)?\s*E(?:D|S)T',head))
   if date and time_ok:
    release=f'{int(date.group(3)):04d}-{mon.index(date.group(1))+1:02d}-{int(date.group(2)):02d}';c['release_date']=release;c['release_time_ET']='10:00';c['first_Qlib_trade_after_release']=next((d for d in calendar if d>release),None)
   start_match=re.search(r'(?im)^\s*Adjusted\s*\d*\s*$',t);adjusted=t[start_match.end():].split('Not Adjusted',1)[0] if start_match else '';matches=list(re.finditer(r'Manufacturers(?:²|3|2)?[^\d]{0,100}?((?:\d[\d,]*\s+){6})(\d+\.\d\d)\s+(\d+\.\d\d)\s+(\d+\.\d\d)',adjusted,re.I|re.S))
   if len(matches)==1:
    mm=matches[0];c['manufacturers_adjusted_current_ratio']=float(mm.group(2));c['manufacturers_adjusted_previous_ratio']=float(mm.group(3));c['same_release_prior_year_ratio']=float(mm.group(4))
   else:c['parse_diagnostic']={'adjusted_Manufacturers_matches':len(matches),'adjusted_chars':len(adjusted)}
   if c['release_date'] and c['first_Qlib_trade_after_release'] and c['manufacturers_adjusted_current_ratio'] is not None:c['status']='source_sample_semantics_qualified'
  case_results.append(c)
status='index_transport_repaired_three_saved_PDF_samples_qualified' if len(case_results)==3 and all(x['status']=='source_sample_semantics_qualified' for x in case_results) else 'index_transport_or_saved_PDF_layout_unqualified'
result={'status':status,'plan_sha256':sha(card/'plan.json'),'index_receipt':receipt,'saved_PDF_case_results':case_results,'index_exact_link_count':sum('https://www2.census.gov/mtis/historical/'+v in links for v in months.values()),'GETs_started':1,'saved_PDF_reads':state['saved_PDF_reads'],'2011_current_bytes_first_public_qualified':False,'full_monthly_panel_qualified':False,'SOXX_outcome_reads':0,'model_fits':0,'native_accounts':0};(card/'result.json').write_text(json.dumps(result,ensure_ascii=False,indent=2)+'\n')
settings=_settings(root);files=sorted(x for x in snap.iterdir() if x.is_file());entries={str(x.relative_to(root)):sha(x) for x in files}
with tempfile.TemporaryDirectory(prefix='mtis-index-repair-') as tmp:
 archive=pathlib.Path(tmp)/'source.tar.gz'
 with tarfile.open(archive,'w:gz') as t:
  for x in files:t.add(x,arcname=str(x.relative_to(root)),recursive=False)
 digest=sha(archive);nas=nas_api_call(origin=settings['origin'],token=settings['token'],method='PUT',path='/api/research/artifacts/'+digest,upload=archive,executor_id='soxl-minute-research')['artifact'];assert nas['sha256']==digest and nas['size_bytes']==archive.stat().st_size;remote=settings['artifact_root'].rstrip('/')+'/'+digest[:2]+'/'+digest[2:4]+'/'+digest;assert _ssh(settings['ssh'],'sha256sum '+shlex.quote(remote)).split()[0]==digest
 with tempfile.TemporaryDirectory(prefix='mtis-index-repair-restore-') as back:
  fetched=pathlib.Path(back)/'source.tar.gz'
  with fetched.open('wb') as output:subprocess.run(['ssh','-p',str(settings['ssh']['port']),'-o','BatchMode=yes',settings['ssh']['host'],'cat '+shlex.quote(remote)],stdout=output,stderr=subprocess.PIPE,check=True,timeout=90)
  assert sha(fetched)==digest
  with tarfile.open(fetched,'r:gz') as t:
   members={m.name:m for m in t.getmembers() if m.isfile()};assert set(members)==set(entries)
   for rel,h in entries.items():assert hashlib.sha256(t.extractfile(members[rel]).read()).hexdigest()==h
source_archive={**nas,'nas_path':remote,'files':entries,'cold_restore_verified':True};(card/'source_archive.json').write_text(json.dumps(source_archive,ensure_ascii=False,indent=2)+'\n')
if not case_results:qlib.init(provider_uri=str(root/'data/qlib/soxl'),region='us',kernels=1)
with R.start(experiment_name=p['experiment'],uri=str(card/'qlib_recorder')):
 rec=R.get_recorder();rec.save_objects(artifact_path='inputs',plan=p,prior_source_archive_sha256=old['source_NAS_sha256'],new_index_archive=source_archive);rec.save_objects(artifact_path='analysis',cases=case_results,result=result);rec.log_metrics(accepted_samples=sum(x['status']=='source_sample_semantics_qualified' for x in case_results),GETs=1,model_fits=0,native_accounts=0);rid,eid=rec.id,str(rec.experiment_id)
state.update(status='finished',recorder_id=rid,experiment_id=eid);(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n');ev=archive_nas_recorder(card/'nas_binding.json',workspace_root=root,card=card,recorder_id=rid,experiment_id=eid);assert _ssh(settings['ssh'],'sha256sum '+shlex.quote(ev['nas_path'])).split()[0]==ev['sha256']
summary={'experiment':p['experiment'],'decision':status,'objective_achieved':False,'plan_sha256':result['plan_sha256'],'result':'result.json','source_archive':'source_archive.json','source_NAS_sha256':digest,'source_NAS_cold_restore_verified':True,'recorder_id':rid,'experiment_id':eid,'recorder_uri':str(card/'qlib_recorder'),'NAS_evidence':ev,'NAS_readback':{'sha256':ev['sha256'],'remote_hash_verified':True},'index_HTTP_status':receipt['http_status'],'qualified_samples':sum(x['status']=='source_sample_semantics_qualified' for x in case_results),'GETs_started':1,'SOXX_outcome_reads':0,'model_fits':0,'native_accounts':0,'artifact_policy':{'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':['inputs/*','analysis/*']},'limitations':['Current 2011 PDF Last-Modified is2014 and does not qualify first-public 2011 byte vintage.','Manufacturers aggregate is not semiconductor-specific and three samples are not a full historical panel.','No Stage1 information test, funded account or SOXX success.']};(card/'summary.json').write_text(json.dumps(summary,indent=2)+'\n');print(json.dumps({'decision':status,'HTTP':receipt['http_status'],'links':result['index_exact_link_count'],'samples':summary['qualified_samples'],'recorder_id':rid}))


2026/10/23 14:43:08 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at /Users/rziiiii/dev/workspace/qlib_venv/lib/python3.12/site-packages/mlflow/assistant/skills/instrumenting-with-mlflow-tracing/SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


[10281:MainThread](2026-10-23 14:43:25,891) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[10281:MainThread](2026-10-23 14:43:25,894) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[10281:MainThread](2026-10-23 14:43:25,894) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/soxl')}


[10281:MainThread](2026-10-23 14:43:25,902) WARNING - qlib.workflow - [expm.py:231] - No valid experiment found. Create a new experiment with name stage1_census_mtis_index_transport_repair_20261023.


[10281:MainThread](2026-10-23 14:43:25,904) INFO - qlib.workflow - [exp.py:258] - Experiment 559934371387262188 starts running ...


[10281:MainThread](2026-10-23 14:43:25,978) INFO - qlib.workflow - [recorder.py:345] - Recorder 5746ff5f0dbe46d8a95ba668c15ace73 starts running under Experiment 559934371387262188 ...


[10281:MainThread](2026-10-23 14:43:26,705) INFO - qlib.timer - [log.py:127] - Time cost: 0.002s | waiting `async_log` Done


{"decision": "index_transport_or_saved_PDF_layout_unqualified", "HTTP": "200", "links": 0, "samples": 0, "recorder_id": "5746ff5f0dbe46d8a95ba668c15ace73"}


# 来源执行结论

本版目录GET为HTTP200，但900,648字节超过700,000字节冻结上限，故没有重读三个PDF，也没有接受数值。后续零新GET的saved_index_size_repair只放宽明确的已存档页面体积并完成三样本核验。

未读SOXX收益，未拟合或运行账户。
